
I spent seven days building/learning:
    - a matrix multiplication kernel in raw CUDA.
    - bridging it to Python through two different interfaces.
    - discovering exactly why my code runs 20× slower than PyTorch. 

This post is what I learned along the way.
The goal wasn't to beat cuBLAS—it was to understand why I couldn't. Every kernel I write in the coming weeks will rest on the mental model I built this week.

## The GPU Mental Model

You cannot write CUDA by translating Python loops into C. A GPU is not a faster CPU. It's a *different* computer entirely.

When we launch a CUDA kernel, the GPU executes the function thousands of times **simultaneously**. Each execution is a **thread**. Threads are grouped into a **block**; blocks form a **grid**. The hierarchy looks like this:

```
Grid
├── Block(0,0) ── Thread(0,0) Thread(0,1) Thread(1,0) ...
├── Block(0,1) ── Thread(0,0) Thread(0,1) ...
├── Block(1,0) ── ...
└── ...
```

Every thread knows exactly two things:
- `blockIdx` — which block in the grid am I in?
- `threadIdx` — which thread within my block am I?

From these, each thread computes its unique coordinates in the output matrix:

```cuda
int row = blockIdx.y * blockDim.y + threadIdx.y;
int col = blockIdx.x * blockDim.x + threadIdx.x;
```

As a kernel writer: use `row` and `col` to do **exactly one piece of work**. No thread collaborates with another by default. They run in parallel isolation.

### The Memory Hierarchy: Why Naive Kernels Are Slow


| Memory | Latency | Size | Scope |
|--------|---------|------|-------|
| Registers | ~1 cycle | Very limited (per thread) | One thread |
| Shared memory | ~4 cycles | 48KB per block | Threads in same block |
| Global memory | ~400 cycles | 16GB+ | All threads |

My kernel reads from global memory repeatedly. cuBLAS reads from shared memory. That 100× latency difference explains nearly everything about the performance gap.

## The Kernel — Line by Line

Here's the complete naive matrix multiplication kernel I wrote, compiled, and debugged:

```cuda
#include <cuda_runtime.h>

__global__ void matmul_kernel(float* A, float* B, float* C,
                              int M, int N, int K) {
    // Each thread computes one element of C
    int row = blockIdx.y * blockDim.y + threadIdx.y;
    int col = blockIdx.x * blockDim.x + threadIdx.x;

    // Guard: skip threads outside matrix bounds
    if (row < M && col < N) {
        float sum = 0.0f;
        for (int i = 0; i < K; i++) {
            sum += A[row * K + i] * B[i * N + col];
        }
        C[row * N + col] = sum;
    }
}

// C interface for Python ctypes
extern "C" {
    void matmul_cuda(float* A, float* B, float* C,
                     int M, int N, int K) {
        dim3 blockDim(16, 16);           // 256 threads per block
        dim3 gridDim((N + 15) / 16,      // Ceiling division for columns
                     (M + 15) / 16);     // Ceiling division for rows
        
        matmul_kernel<<<gridDim, blockDim>>>(A, B, C, M, N, K);
        cudaDeviceSynchronize();         // Wait for GPU to finish
    }
}
```

Let me explain what took me hours to truly internalize:

**`__global__`** : This function runs on the GPU but is called from the CPU. Every thread enters here simultaneously.

**The coordinate calculation** : This is the heart of CUDA. `blockIdx.y * blockDim.y + threadIdx.y` maps the 2D grid of threads onto the 2D matrix of outputs. I spent an hour drawing 4×4 grids on paper until I could trace any `(blockIdx, threadIdx)` pair to its output element without thinking.

**The guard condition** : Matrix dimensions rarely divide evenly by 16. If N=33, I need 3 blocks (48 threads) to cover 33 columns. The last 15 threads would compute garbage or worse, write to invalid memory. The guard makes them exit early.

**Row-major indexing** : CUDA has no 2D arrays. `A[row * K + i]` treats a flat 1D buffer as a matrix. For a 3×3 matrix, element [1][2] lives at index `1*3 + 2 = 5`. I verified this with pen and paper until it felt automatic.

**Ceiling division** — `(N + 15) / 16` instead of `N / 16`. Integer division truncates; I need to round up to cover all columns. The `+15` ensures we get the next integer when there's any remainder.

**`extern "C"`** — C++ mangles function names (encoding types into the symbol). `matmul_cuda` becomes something like `_Z12matmul_cudaPfS_S_iii`. ctypes looks up functions by exact string match. `extern "C"` disables name mangling so Python can find the symbol.

**`cudaDeviceSynchronize()`** — The CPU launches the kernel and returns immediately. Without synchronization, the CPU might read `C` before the GPU finishes writing it. I learned this the hard way: tests passed randomly, then failed randomly, until I added this line.

Compile with:
```bash
nvcc -shared -o kernels/matmul.so kernels/matmul.cu \
     -Xcompiler -fPIC -O2
```

## Two Bridges from Python to CUDA

Getting the kernel to run was only half the battle. I needed to call it from Python. I built two bridges: the manual way and the production way.

In [ ]:
import ctypes
import torch

# Load the compiled shared library
matmul_lib = ctypes.CDLL("./kernels/matmul.so")

# CRITICAL: Declare argument types. Python has no idea what the C 
# function expects. Without this, it passes garbage.
matmul_lib.matmul_cuda.argtypes = [
    ctypes.POINTER(ctypes.c_float),  # float* A
    ctypes.POINTER(ctypes.c_float),  # float* B
    ctypes.POINTER(ctypes.c_float),  # float* C
    ctypes.c_int, ctypes.c_int, ctypes.c_int  # M, N, K
]
matmul_lib.matmul_cuda.restype = None  # Returns void

def matmul_cuda(A: torch.Tensor, B: torch.Tensor) -> torch.Tensor:
    # Defensive checks: fail loudly, not silently
    assert A.is_cuda and B.is_cuda, "Tensors must be on GPU"
    assert A.dtype == torch.float32, "Only float32 supported"
    assert A.is_contiguous() and B.is_contiguous(), "Must be contiguous"
    
    M, K = A.shape
    K2, N = B.shape
    assert K == K2, f"Shape mismatch: {M}×{K} @ {K2}×{N}"
    
    C = torch.zeros(M, N, device="cuda", dtype=torch.float32)
    
    # Get raw GPU memory addresses and cast to C pointers
    A_ptr = ctypes.cast(A.data_ptr(), ctypes.POINTER(ctypes.c_float))
    B_ptr = ctypes.cast(B.data_ptr(), ctypes.POINTER(ctypes.c_float))
    C_ptr = ctypes.cast(C.data_ptr(), ctypes.POINTER(ctypes.c_float))
    
    matmul_lib.matmul_cuda(A_ptr, B_ptr, C_ptr, M, N, K)
    return C

Three things I had to handle manually:

1. **`argtypes` declaration** — ctypes defaults to passing integers. Without explicit types, my float pointers were interpreted as integers. The function received garbage addresses and either segfaulted or returned nonsense.

2. **`data_ptr()` and casting** — `A.data_ptr()` returns the GPU memory address as a Python integer. ctypes needs a typed pointer, so I cast it. This is the exact boundary between Python objects and raw memory.

3. **The `is_contiguous()` trap** — PyTorch tensors can be non-contiguous (e.g., after `.transpose()`). They share memory but use different indexing. My kernel assumes row-major sequential layout. A non-contiguous tensor would pass all shape checks but compute wrong results **silently**. I spent an hour debugging a "correct" test before realizing the tensor was transposed.

### Bridge 2: cpp_extension (The Production Path)

Once I understood what ctypes was doing manually, I switched to PyTorch's `cpp_extension`, which automates everything I just struggled through.

```cuda
// kernels/matmul_ext.cu
#include <torch/extension.h>

__global__ void matmul_kernel(float* A, float* B, float* C,
                              int M, int N, int K) {
    // ... same kernel as before ...
}

torch::Tensor matmul_cuda_ext(torch::Tensor A, torch::Tensor B) {
    TORCH_CHECK(A.is_cuda() && B.is_cuda(), "Tensors must be on GPU");
    TORCH_CHECK(A.dtype() == torch::kFloat32, "Support Float32 only"
    TORCH_CHECK(A.is_contiguous() && B.is_contiguous(), "Must be contiguous");
    
    int M = A.size(0), K = A.size(1), N = B.size(1);
    auto C = torch::zeros({M, N}, A.options());
    
    dim3 blockDim(16, 16);
    dim3 gridDim((N + 15) / 16, (M + 15) / 16);
    
    matmul_kernel<<<gridDim, blockDim>>>(
        A.data_ptr<float>(),
        B.data_ptr<float>(),
        C.data_ptr<float>(),
        M, N, K
    );
    cudaDeviceSynchronize();
    return C;
}

PYBIND11_MODULE(TORCH_EXTENSION_NAME, m) {
    m.def("matmul", &matmul_cuda_ext, "Matrix multiply (CUDA)");
}
```

In [ ]:
# src/kernel_ext.py
from torch.utils.cpp_extension import load

matmul_ext = load(
    name="matmul_ext",
    sources=["kernels/matmul_ext.cu"],
    verbose=True  # Shows compilation output on first run
)

def matmul_cuda_v2(A, B):
    return matmul_ext.matmul(A, B)

No manual `nvcc`. No `argtypes`. No `ctypes.cast()`. PyTorch compiles on first import, caches the result, and handles type conversion automatically. `TORCH_CHECK` provides clean error messages integrated with PyTorch's exception system.

### Why I Needed Both

| Aspect | ctypes | cpp_extension |
|--------|--------|---------------|
| Compilation | Manual `nvcc` command | Automatic on first import |
| Type safety | Manual `argtypes` declaration | C++ type system |
| Pointer handling | `data_ptr()` + `ctypes.cast()` | `tensor.data_ptr<float>()` |
| Error messages | Python `assert` | `TORCH_CHECK` macros |
| Used in production | No | Yes (vLLM, FlashAttention, etc.) |

I needed ctypes to understand what cpp_extension hides. Now I use cpp_extension, but I know exactly what it's doing underneath.

## Benchmarking: Measuring Performance

GPU timing is subtle. `time.time()` measures **CPU** time—the moment the kernel was launched, not when it finished. GPU operations are asynchronous.

The correct method uses CUDA Events, which are timestamps recorded on the GPU itself:

In [ ]:
def benchmark(fn, A, B, n=100, warmup=10):
    # Warmup: first run triggers JIT compilation
    for _ in range(warmup):
        fn(A, B)
    torch.cuda.synchronize()
    
    # Actual timing with CUDA Events
    start = torch.cuda.Event(enable_timing=True)
    end = torch.cuda.Event(enable_timing=True)
    
    start.record()
    for _ in range(n):
        fn(A, B)
    end.record()
    torch.cuda.synchronize()
    
    return start.elapsed_time(end) / n  # milliseconds

Results on a T4 GPU:

| Matrix Size | ctypes (ms) | cpp_ext (ms) |Pytorch |
|-------------|-------------|--------------|---------------------|
| 128×128×128 | 0.053 ms | 0.053 ms | 0.015 ms |
| 512×512×512 | 0.902 ms | 0.494 ms | 0.055 ms |
| 1024×1024×1024 | 4.382 ms | 4.384 ms | 0.646 ms |

To interpret these, compute **GFLOP/s** (billions of floating-point operations per second):

$$\text{GFLOP/s} = \frac{2 \times M \times K \times N}{\text{time (s)} \times 10^9}$$

A T4 has a theoretical peak of ~65 TFLOP/s. PyTorch typically achieves 40–60% of peak on large matrices. My kernel achieves a tiny fraction. The gap is 10–50×, and now I know exactly why.

## Why My Kernel Is Slow 

My naive implementation reads from **global memory** on every iteration of the inner loop. Global memory latency is ~400 cycles. For a 1024×1024 matmul, each output element requires 2048 global memory reads (1024 from A, 1024 from B). That's over 800,000 cycles of memory latency per element.

cuBLAS uses **shared memory tiling**:

1. Load a 16×16 (or 32×32) tile of A and B from global memory into shared memory (~4 cycles)
2. Compute partial results for all elements in the tile using fast shared memory
3. Load the next tile, accumulate, repeat

This reduces global memory accesses by the tile size factor (16–32×). That's where the performance gap comes from.

I didn't implement tiling this week. That was intentional. The goal was to understand *why* the gap exists before trying to close it. Next week, I'll add shared memory tiling and watch the numbers move.

## Where This Fits: Matmul in Llama

Every matmul I wrote this week corresponds to a real operation in Llama's transformer block:

```python
# In every attention layer:
Q = x @ W_q          # (seq_len, dim) @ (dim, n_heads * head_dim)
K = x @ W_k          # Same for keys
V = x @ W_v          # Same for values
scores = Q @ K.T     # (seq_len, head_dim) @ (head_dim, seq_len) = O(seq_len²)
output = scores @ V  # (seq_len, seq_len) @ (seq_len, head_dim)
```

The `Q @ K.T` operation is the infamous quadratic scaling bottleneck. Double the context length, quadruple the compute. This is exactly what FlashAttention, paged attention, and speculative decoding optimize.

The kernel I wrote this week is the primitive that makes all of those optimizations possible. I needed to feel its limitations in my bones before I can transcend them.

## What I Learned

1. **CUDA is not parallel Python**. It's a data-parallel execution model where you map threads to data, not instructions to execution order.

2. **Memory hierarchy dominates performance**. Algorithmic complexity (O(n³) for matmul) is necessary but not sufficient. Memory access patterns determine actual speed.

3. **The boundary between Python and C is explicit**. `data_ptr()` is the gateway. Everything else is convention and error checking.

4. **Correctness comes from invariants, not hope**. The `is_contiguous()` check, the guard condition, the synchronization—these aren't optional niceties. They're load-bearing structural elements.

5. **Slow and understood beats fast and mysterious**. My kernel is 20× slower than cuBLAS, but I can explain every cycle of that gap. Next week, I'll start closing it.

## What's Next

**Week 2**: Shared memory tiling. I'll rewrite the kernel to load tiles into `__shared__` memory, synchronize threads with `__syncthreads()`, and watch the benchmark numbers improve. Then I'll plug the kernel into an actual attention layer.

The foundation is built. The fun starts now.

---

**Repository**: [mini-inference-engine](https://github.com/iSmailTG/Cuda/tree/main/mini-inference-engine)  
**Full code**: All kernels, bridges, tests, and benchmarks are in the repo.